ref: https://cellcharter.readthedocs.io/en/latest/notebooks/cosmx_human_nsclc.html#cellcharter-s-spatial-clustering

In [1]:
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import os

import kktk

import cellcharter

/home/kk837/.conda/envs/cellcharter_env/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/kk837/.conda/envs/cellcharter_env/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/kk837/.conda/envs/cellcharter_env/lib/python3.12/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_at

In [2]:
import squidpy as sq

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import session_info
session_info.show()

In [5]:
sc.settings.set_figure_params(dpi=80)

# Function

# Variables

In [6]:
data_object_dir = '/rfs/project/rfs-iCNyzSAaucw/kk837/data_objects/Foetal/VisiumHD/Revision_Oct2025'
path_adata = f'{data_object_dir}/all_b2c_cells_filtered_raw.h5ad' # use the object which contains all the qualified cells
library_key='section_ID'
latent_space = 'scVI_latent_n-layers-3'
cluster_col = 'cluster_cellcharter_layer-0123_k-8'

In [7]:
os.makedirs(f'{data_object_dir}/cellcharter_cluster_subsets', exist_ok=True)

# Read in adata

In [8]:
adata = sc.read_h5ad(path_adata)
adata

AnnData object with n_obs × n_vars = 225097 × 18085
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'scVI_latent_n-layers-3_leiden_0.2', 'scVI_latent_n-layers-3_leiden_0.3', 'scVI_latent_n-layers-3_leiden_0.4', 'scVI_latent_n-layers-3_leiden_0.5', 'scVI_latent_n-layers-3_leiden_0.8', 'scVI_latent_n-layers-3_leiden_1.0', 'scVI_latent_n-layers-3_leiden_1.5', 'scVI_latent_n-layers-3_leiden_2.0', 'scVI_latent_n-layers-3_leiden_3.0', 'scVI_latent_n-layers-3_leiden_4.0', 'coarse_grain_pre', 'coarse_grain_pre_2', 'coarse_grain_tacco_ref-hd-sc

In [9]:
adata.X.data[:5]

array([1., 1., 1., 1., 1.], dtype=float32)

In [10]:
# remove previously calculated cellcharter-related outputs
## obs
obs_col_keep = [x for x in adata.obs.columns if 'cluster_cellcharter' not in x]
obs_col_keep = obs_col_keep + [cluster_col] # add the selected cluster_col
adata.obs = adata.obs[obs_col_keep].copy()

## obsm
keys = list(adata.obsm.keys()).copy()
for key in keys:
    if 'cellcharter' in key:
        del adata.obsm[key]

## obsp
keys = list(adata.obsp.keys()).copy()
for key in keys:
    if 'cc_' in key:
        del adata.obsp[key]

adata

AnnData object with n_obs × n_vars = 225097 × 18085
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'scVI_latent_n-layers-3_leiden_0.2', 'scVI_latent_n-layers-3_leiden_0.3', 'scVI_latent_n-layers-3_leiden_0.4', 'scVI_latent_n-layers-3_leiden_0.5', 'scVI_latent_n-layers-3_leiden_0.8', 'scVI_latent_n-layers-3_leiden_1.0', 'scVI_latent_n-layers-3_leiden_1.5', 'scVI_latent_n-layers-3_leiden_2.0', 'scVI_latent_n-layers-3_leiden_3.0', 'scVI_latent_n-layers-3_leiden_4.0', 'coarse_grain_pre', 'coarse_grain_pre_2', 'coarse_grain_tacco_ref-hd-sc

# Log-normalise

In [11]:
adata_lognorm = adata.copy()

# log-normalise the count
sc.pp.filter_genes(adata_lognorm, min_counts=3)
sc.pp.normalize_total(adata_lognorm, target_sum=1e4)
sc.pp.log1p(adata_lognorm)
print(adata_lognorm.X.data[:5])

[1.8897898 1.8897898 1.8897898 1.8897898 1.8897898]


# Detect spatial neighbors

In [12]:
# detect spatial neighbors
sq.gr.spatial_neighbors(adata_lognorm, spatial_key='spatial',
                        library_key='section_ID', coord_type='generic', delaunay=True, percentile=99)
# the Delaunay triangulation has the drawback of generating connections between distant cells.
# We remove this long links using CellCharter’s gr.remove_long_links function.
cellcharter.gr.remove_long_links(adata_lognorm)

In [13]:
adata_lognorm

AnnData object with n_obs × n_vars = 225097 × 18070
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'scVI_latent_n-layers-3_leiden_0.2', 'scVI_latent_n-layers-3_leiden_0.3', 'scVI_latent_n-layers-3_leiden_0.4', 'scVI_latent_n-layers-3_leiden_0.5', 'scVI_latent_n-layers-3_leiden_0.8', 'scVI_latent_n-layers-3_leiden_1.0', 'scVI_latent_n-layers-3_leiden_1.5', 'scVI_latent_n-layers-3_leiden_2.0', 'scVI_latent_n-layers-3_leiden_3.0', 'scVI_latent_n-layers-3_leiden_4.0', 'coarse_grain_pre', 'coarse_grain_pre_2', 'coarse_grain_tacco_ref-hd-sc

# Save per cellcharter cluster

In [14]:
# delete uns.spatial to save file size
del adata_lognorm.uns['spatial']

In [15]:
set(adata_lognorm.obs[cluster_col])

{'0', '1', '2', '3', '4', '5', '6', '7'}

In [16]:
for cluster in set(adata_lognorm.obs[cluster_col]):
    print(f'cluster {cluster}')
    
    # subset
    adata_sub = adata_lognorm[adata_lognorm.obs[cluster_col]==cluster].copy()
    print(f"shape of subsetted adata: {adata_sub.shape}")
    print(f"shape of spatial_connectivities: {adata_sub.obsp['spatial_connectivities'].shape}")
    print(f"shape of spatial_distances: {adata_sub.obsp['spatial_distances'].shape}")
    
    # save
    adata_sub.write(f'{data_object_dir}/cellcharter_cluster_subsets/cc_cluster{cluster}_lognorm.h5ad')
    del adata_sub
    print('')

cluster 7
shape of subsetted adata: (27282, 18070)
shape of spatial_connectivities: (27282, 27282)
shape of spatial_distances: (27282, 27282)

cluster 2
shape of subsetted adata: (37257, 18070)
shape of spatial_connectivities: (37257, 37257)
shape of spatial_distances: (37257, 37257)

cluster 3
shape of subsetted adata: (25658, 18070)
shape of spatial_connectivities: (25658, 25658)
shape of spatial_distances: (25658, 25658)

cluster 5
shape of subsetted adata: (21111, 18070)
shape of spatial_connectivities: (21111, 21111)
shape of spatial_distances: (21111, 21111)

cluster 0
shape of subsetted adata: (46916, 18070)
shape of spatial_connectivities: (46916, 46916)
shape of spatial_distances: (46916, 46916)

cluster 6
shape of subsetted adata: (25892, 18070)
shape of spatial_connectivities: (25892, 25892)
shape of spatial_distances: (25892, 25892)

cluster 1
shape of subsetted adata: (13892, 18070)
shape of spatial_connectivities: (13892, 13892)
shape of spatial_distances: (13892, 13892)


In [17]:
!ls -lh {data_object_dir}/cellcharter_cluster_subsets

total 1.3G
-rwxrwx---+ 1 kk837 kk837 455M Feb 19 22:24 cc_cluster0_lognorm.h5ad
-rwxrwx---+ 1 kk837 kk837 176M Feb 19 22:24 cc_cluster1_lognorm.h5ad
-rwxrwx---+ 1 kk837 kk837 266M Feb 19 22:24 cc_cluster2_lognorm.h5ad
-rwxrwx---+ 1 kk837 kk837 266M Feb 19 22:24 cc_cluster3_lognorm.h5ad
-rwxrwx---+ 1 kk837 kk837 226M Feb 19 22:24 cc_cluster4_lognorm.h5ad
-rwxrwx---+ 1 kk837 kk837 206M Feb 19 22:24 cc_cluster5_lognorm.h5ad
-rwxrwx---+ 1 kk837 kk837 242M Feb 19 22:24 cc_cluster6_lognorm.h5ad
-rwxrwx---+ 1 kk837 kk837 258M Feb 19 22:24 cc_cluster7_lognorm.h5ad
drwxrwx---+ 2 kk837 kk837    0 Feb 19 22:11 cellcharter_autok
